# 2D Pseudo-Differential Operator Analysis with `psiop`

This notebook is the two-dimensional counterpart of `1D_symbol.ipynb`. It analyses a 2D pseudo-differential operator on phase space $T^*\mathbb{R}^2 \ni (x, y, \xi, \eta)$.

We will:
1. Explore the symbol interactively with `interactive_symbol_analysis` (2D mode).
2. Inspect the **symbol anatomy**: Peetre decomposition, principal symbol, order, homogeneity, ellipticity, Hamiltonian field. *(This replaces `pseudospectrum_analysis`, which is 1D-only and raises `NotImplementedError` for `dim != 1`.)*
3. Integrate and visualise the **Hamiltonian (bicharacteristic) flow** in 2D.
4. Apply the operator to a family of **2D test fields** (Gaussian, off-centre bump, chirp, WKB packet, vortex).
5. Evolve $u_t = i P u$ with `solve_first_order` and inspect the space-time field, $L^2$ mass conservation, and an animation.

> **Sign convention.** `psiop` uses
> $$u(x)=\int e^{i x\cdot\xi}\,\hat u(\xi)\,\frac{d\xi}{(2\pi)^d}, \qquad (Pu)(x)=\int e^{i x\cdot\xi}\,p(x,\xi)\,\hat u(\xi)\,\frac{d\xi}{(2\pi)^d}$$
> so a polynomial symbol maps to a differential operator via $\xi\mapsto -i\partial_x$, $\eta\mapsto -i\partial_y$. In particular $\xi^2+\eta^2 \mapsto -\Delta$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from IPython.display import HTML

# Assuming psiop is on the python path
from psiop import *

# Spatial and frequency variables
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

# ------------------------------------------------------------------
# Candidate 2D symbols -- uncomment to experiment
# ------------------------------------------------------------------
# symbol_expr = xi**2 + eta**2                                      # free -Laplacian (straight rays)
# symbol_expr = y * xi - x *eta                                     # vortex
# symbol_expr = (xi**2 + eta**2) * (1 + sp.cos(x)*sp.cos(y))        # variable-coefficient Laplacian
# symbol_expr = sp.I*(xi*sp.cos(x) + eta*sp.sin(y))                 # anisotropic transport
# symbol_expr = sp.sqrt(xi**2 + eta**2 + 1)                         # relativistic / order-1 elliptic
# symbol_expr = sp.exp(sp.I*(x*xi + y*eta))*(xi**2 + eta**2)        # NUFFT-representable joint term
# symbol_expr = 1/(1 + (1 - xi)**2 + (1 - eta)**2)                  # AAA-representable joint term
symbol_expr = xi**2 + eta**2 + sp.cos(x)*sp.cos(y)                # -Laplacian + periodic potential

op = PseudoDifferentialOperator(
    expr=symbol_expr,
    vars_x=[x, y],
    mode='symbol',
    quantization='kohn-nirenberg',
    apply_backend='peetre',
)

print('Operator symbol:')
sp.pprint(op.symbol)
print(f'\ndimension           : {op.dim}')
print(f'quantization        : {op.quantization}')
print(f'apply backend       : {op.apply_backend}')
print(f'spatially dependent : {op._is_spatial_dependent()}')


## 1. Interactive Symbol Analysis (2D)

`interactive_symbol_analysis` launches an `ipywidgets` dashboard. In 2D it exposes **four** sliders -- $\xi_0$, $\eta_0$, $x_0$, $y_0$ -- and the modes

`Symbol Amplitude`, `Symbol Phase`, `Micro-Support (1/|p|)`, `Cotangent Fiber`, `Characteristic Set`, `Characteristic Gradient`, `Symplectic Vector Field`, `Hamiltonian Flow`.

Only the sliders relevant to the currently selected mode are displayed (the `needs` table drives this).

> **Note:** in a static JSON viewer the widgets will not render. Run this notebook in a live Jupyter environment to interact with the sliders.


In [ ]:
# Launch the interactive dashboard (2D branch: xlim/ylim for position, xi_range/eta_range for momentum)
op.interactive_symbol_analysis(
    xlim=(-6, 6),          # x-slider range
    ylim=(-6, 6),          # y-slider range
    xi_range=(-10, 10),    # xi-slider range
    eta_range=(-10, 10),   # eta-slider range
    density=40
)


## 2. Symbol Anatomy & Microlocal Properties

`pseudospectrum_analysis` is **1D only** (it raises `NotImplementedError('Pseudospectrum analysis currently supports 1D only')`), so this section replaces it with the 2D-appropriate microlocal diagnostics:

| diagnostic | call |
|---|---|
| Peetre splitting `local / separable / joint residual` | `op.print_peetre_decomposition()` |
| principal symbol $p_m(x,\xi)$ | `op.principal_symbol(order=1)` |
| asymptotic order $m$ | `op.symbol_order()` |
| homogeneity in $(\xi,\eta)$ | `op.is_homogeneous()` |
| expansion in $\rho=\sqrt{\xi^2+\eta^2}$ | `op.asymptotic_expansion(order=3)` |
| Hamiltonian vector field $H_p$ | `op.symplectic_flow()` |
| ellipticity in the high-frequency region | `op.is_elliptic_numerically(...)` |


In [ ]:
%%time
print('=' * 72)
print('PEETRE DECOMPOSITION   (local  +  separable  +  joint residual)')
print('=' * 72)
# For dim=2 every additive term a(x,y)*q(xi,eta) with q polynomial in (xi,eta)
# is classified as *local*, i.e. a differential operator with variable coefficients.
op.print_peetre_decomposition()

print('\n' + '=' * 72)
print('ASYMPTOTIC / MICROLOCAL DATA')
print('=' * 72)
print('principal symbol (order 1) : ', end='')
sp.pprint(op.principal_symbol(order=1))
print('symbol order               :', op.symbol_order())
print('homogeneous?               :', op.is_homogeneous())
print('\nasymptotic expansion (order 3) in rho = sqrt(xi^2 + eta^2):')
sp.pprint(op.asymptotic_expansion(order=3))

print('\nHamiltonian (symplectic) vector field H_p:')
for k, v in op.symplectic_flow().items():
    print(f'  {k:8s} = ', end='')
    sp.pprint(v)

# Numerical ellipticity test. For dim=2 the grids must be passed as
# (x_array, y_array) and (xi_array, eta_array).
xg_e = np.linspace(-6, 6, 64)
kg_e = np.linspace(-25, 25, 64)
print('\nis_elliptic_numerically(order=2) :',
      op.is_elliptic_numerically((xg_e, xg_e), (kg_e, kg_e),
                                 order=2, n_random=20000, seed=0))


### 2.1 Static slices of the symbol

Two complementary 2D slices (these are the non-interactive versions of what the widget shows, so they render in any viewer):

* **position slice** $|p(x,y,\xi_0,\eta_0)|$ -- the coefficient landscape seen by a wave packet carrying a fixed momentum;
* **frequency slice / cotangent fibre** $|p(x_0,y_0,\xi,\eta)|$ -- whose small level sets form the **characteristic set** $\Sigma = \{p \approx 0\}$. Rays of $P$ propagate along $\Sigma$.


In [ ]:
lin_s = np.linspace(-6, 6, 240)      # spatial axis
lin_k = np.linspace(-8, 8, 240)      # frequency axis

xi0, eta0 = 2.0, 1.0                 # frozen momentum
x0, y0    = 1.0, 0.5                 # frozen position

Xs, Ys = np.meshgrid(lin_s, lin_s, indexing='ij')
Ks, Es = np.meshgrid(lin_k, lin_k, indexing='ij')

# p_func is lambdified over (x, y, xi, eta)
Z_space = op.p_func(Xs, Ys, xi0, eta0)     # p(x, y, xi0, eta0)
Z_fiber = op.p_func(x0, y0, Ks, Es)        # p(x0, y0, xi, eta)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# --- position slice -------------------------------------------------
pcm = axes[0].pcolormesh(Xs, Ys, np.abs(Z_space), shading='auto', cmap='inferno')
fig.colorbar(pcm, ax=axes[0], label='$|p|$')
axes[0].set_title(rf'Position slice  $|p(x,y,\xi_0={xi0},\eta_0={eta0})|$')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y'); axes[0].set_aspect('equal')

# --- cotangent fibre ------------------------------------------------
cf = axes[1].contourf(Ks, Es, np.abs(Z_fiber), levels=50, cmap='viridis')
fig.colorbar(cf, ax=axes[1], label='$|p|$')
axes[1].set_title(rf'Cotangent fibre at $(x_0,y_0)=({x0},{y0})$')
axes[1].set_xlabel(r'$\xi$'); axes[1].set_ylabel(r'$\eta$'); axes[1].set_aspect('equal')

# --- characteristic set ---------------------------------------------
axes[2].contourf(Ks, Es, np.abs(Z_fiber), levels=50, cmap='viridis', alpha=0.35)
axes[2].contour(Ks, Es, np.abs(Z_fiber), levels=[0.25], colors='red', linewidths=2.2)
axes[2].contour(Ks, Es, np.real(Z_fiber), levels=[0.0], colors='white',
                linewidths=1.3, linestyles='--')
axes[2].set_title(r'Characteristic set $\Sigma=\{|p|\approx 0\}$')
axes[2].set_xlabel(r'$\xi$'); axes[2].set_ylabel(r'$\eta$')
axes[2].grid(True, alpha=0.4); axes[2].set_aspect('equal')

fig.suptitle('2D symbol slices', y=1.03, fontsize=14)
fig.tight_layout()
plt.show()


## 3. Hamiltonian (Bicharacteristic) Flow

Wave-front / singularity propagation of $P$ is governed by the Hamilton flow of $H = \operatorname{Re} p$:

$$\dot x = \partial_\xi H, \qquad \dot y = \partial_\eta H, \qquad \dot\xi = -\partial_x H, \qquad \dot\eta = -\partial_y H.$$

For $p = \xi^2+\eta^2+\cos x\cos y$ this reads

$$\dot x = 2\xi,\qquad \dot y = 2\eta,\qquad \dot\xi = \sin x\,\cos y,\qquad \dot\eta = \cos x\,\sin y,$$

i.e. genuinely **curved** rays (the free Laplacian would give straight lines, since $\dot\xi=\dot\eta=0$). `plot_hamiltonian_flow` integrates one ray with `solve_ivp` and overlays the position-space velocity field $(\dot x,\dot y)$; `animate_singularity` animates the same trajectory.


In [ ]:
# A single ray launched from (x0, y0) with initial momentum (xi0, eta0)
op.plot_hamiltonian_flow(
    x0=-3.0, y0=-1.0,
    xi0=2.0, eta0=1.0,
    tmax=8.0,
    n_steps=600,
    show_field=True,
)


In [ ]:
# Animate the same ray in position space (projection='position' is the 2D default)
anim_flow = op.animate_singularity(
    x0=-3.0, y0=-1.0,
    xi0=2.0, eta0=1.0,
    tmax=8.0,
    n_frames=200,
)
HTML(anim_flow.to_jshtml())


## 4. Application to 2D Test Fields

We apply $P$ to a family of two-dimensional states:

1. **Constant** $u=1$ -- annihilated by $-\Delta$, so only the potential $\cos x\cos y$ survives.
2. **Gaussian bump** $e^{-(x^2+y^2)/2}$ -- smooth, band-limited.
3. **Off-centre bump** $e^{-((x-1.5)^2+(y+1)^2)}$ -- probes spatial localisation against $\cos x\cos y$.
4. **Chirp** $e^{\tfrac{i}{2}(x^2+y^2)}e^{-0.1(x^2+y^2)}$ -- quadratic phase / dispersion.
5. **WKB packet** $e^{i(12x+8y)}e^{-((x-1)^2+(y-1)^2)/2}$ -- high-frequency transport; semiclassically $Pu\approx p(x_0,12,8)\,u$.
6. **Vortex** $(x+iy)e^{-(x^2+y^2)/2}$ -- angular momentum / phase singularity at the origin.

All applications use a **periodic** grid with the `peetre` backend. (In 2D, `boundary_condition='dirichlet'` routes through `kohn_nirenberg_nonperiodic`, whose direct quadrature is $\mathcal{O}(N^4)$ -- avoid it here.)


In [ ]:
%%time
# ---------------- grid (indexing='ij' -> fields have shape (Nx, Ny)) ----
L, N = 6.0, 128
x_app = np.linspace(-L, L, N, endpoint=False)
y_app = np.linspace(-L, L, N, endpoint=False)
dx, dy = x_app[1] - x_app[0], y_app[1] - y_app[0]
kx = 2.0 * np.pi * np.fft.fftfreq(N, d=dx)
ky = 2.0 * np.pi * np.fft.fftfreq(N, d=dy)
X, Y = np.meshgrid(x_app, y_app, indexing='ij')

# ---------------- test fields -------------------------------------------
def u_constant(X, Y): return np.ones_like(X, dtype=complex)
def u_gauss(X, Y):    return np.exp(-0.5 * (X**2 + Y**2))
def u_bump(X, Y):     return np.exp(-((X - 1.5)**2 + (Y + 1.0)**2))
def u_chirp(X, Y):    return np.exp(0.5j * (X**2 + Y**2)) * np.exp(-0.1 * (X**2 + Y**2))
def u_wkb(X, Y):      return np.exp(1j * (12.0*X + 8.0*Y)) * np.exp(-0.5*((X-1.0)**2 + (Y-1.0)**2))
def u_vortex(X, Y):   return (X + 1j*Y) * np.exp(-0.5 * (X**2 + Y**2))

test_functions = {
    'Constant':                 u_constant(X, Y),
    'Gaussian bump':            u_gauss(X, Y),
    'Off-centre bump':          u_bump(X, Y),
    'Chirp':                    u_chirp(X, Y),
    'WKB (highly oscillatory)': u_wkb(X, Y),
    'Vortex':                   u_vortex(X, Y),
}

col_titles = [r'$|u|$', r'$\mathrm{Re}(Pu)$', r'$\mathrm{Im}(Pu)$']
cmaps      = ['inferno', 'RdBu_r', 'RdBu_r']

fig, axes = plt.subplots(len(test_functions), 3,
                         figsize=(15.5, 4.2 * len(test_functions)))

for i, (name, u) in enumerate(test_functions.items()):
    Pu = op.apply(
        u,
        x_grid=x_app, kx=kx,
        y_grid=y_app, ky=ky,
        boundary_condition='periodic',
        freq_window='gaussian',
        clamp=1e6,
        backend='peetre',
        joint_backend='auto',
        joint_degree=6,
        joint_tol=1e-5,
        joint_max_rel_error=1e-5,
    )

    panels = [np.abs(u), np.real(Pu), np.imag(Pu)]
    for j, (Z, cm) in enumerate(zip(panels, cmaps)):
        ax = axes[i, j]
        # Z has shape (Nx, Ny); pcolormesh(x, y, C) wants C of shape (Ny, Nx)
        pcm = ax.pcolormesh(x_app, y_app, Z.T, shading='auto', cmap=cm)
        if j > 0:                       # symmetric limits for signed panels
            v = float(np.max(np.abs(Z))) or 1.0
            pcm.set_clim(-v, v)
        fig.colorbar(pcm, ax=ax)
        ax.set_aspect('equal')
        ax.set_xlabel('x'); ax.set_ylabel('y')
        if j == 0:
            ax.text(-0.34, 0.5, name, transform=ax.transAxes,
                    rotation=90, va='center', ha='center',
                    fontsize=12, fontweight='bold')

for j, ct in enumerate(col_titles):
    axes[0, j].set_title(ct, fontsize=15)

fig.suptitle('Action of $P$ on 2D test fields', y=1.001, fontsize=16)
fig.tight_layout()
plt.show()


## 5. Time Evolution  $u_t = i\,P u$

`solve_first_order` builds the exponential propagator symbol once, **symbolically**,

$$E(\Delta t) \;=\; \exp\!\big(\Delta t\, p\big) \;=\; \sum_{n\le \text{order}} \frac{(\Delta t)^n}{n!}\, p^{\circ n},$$

where $p^{\circ n}$ is the $n$-fold **asymptotic Kohn-Nirenberg composition** (so the $x$-dependence of the coefficients is handled correctly, not just pointwise multiplication). It then applies $E(\Delta t)$ by FFT at every step.

Because $p = \xi^2+\eta^2+\cos x\cos y$ is **real**, the operator $P \mapsto -\Delta + \cos x\cos y$ is self-adjoint and $u_t = iPu$ is a dispersive, $L^2$-conserving Schrodinger-type flow. Evolving $u_t = Pu$ instead would be a backward heat equation and blow up immediately.

> **2D API differences.** `solve_first_order` calls the initial-condition function as `f(X, Y)` with the *full meshgrids*, and returns `grids = (x_grid, y_grid, kx, ky)`.

> The first run pays a one-time symbolic cost to build $E(\Delta t)$; `build_propagator` caches it per `(symbol, order, quantization, mode, backend)`. Lower `order` to 2 if the build feels slow.


In [ ]:
%%time
# 1. Initial condition: off-centre Gaussian packet with a carrier momentum
def f_initial(X, Y):
    return (np.exp(-0.5 * ((X - 1.5)**2 + (Y + 1.0)**2))
          #  * np.exp(1j * (3.0*X - 2.0*Y))
           )

# 2. Time-stepping parameters
dt         = 0.002
n_steps    = 1500      # t_max = 3.0
save_every = 15        # -> 101 stored frames

# 3. Evolve  du/dt = i * P u   (dispersive / Schrodinger-like)
t_arr, U_arr, grids = solve_first_order(
    s_expr        = sp.I * op.symbol,
    vars_x        = [x, y],
    f             = f_initial,
    dt            = dt,
    n_steps       = n_steps,
    order         = 3,
    L             = 6.0,
    N             = 128,
    save_every    = save_every,
    apply_backend = 'peetre',
)
x_grid, y_grid, kx_grid, ky_grid = grids

print('stored frames :', U_arr.shape[0])
print('field shape   :', U_arr.shape[1:], '  (Nx, Ny)')
print('t in [{:.3f}, {:.3f}]'.format(t_arr[0], t_arr[-1]))


In [ ]:
# Space-time snapshots of the amplitude
plot_scalar_2d(t_arr, U_arr, x_grid, y_grid, quantity='abs')


In [ ]:
# Space-time snapshots of the real part
plot_scalar_2d(t_arr, U_arr, x_grid, y_grid, quantity='real')


### 5.1 $L^2$ mass conservation (end-to-end validation)

Since $-\Delta+\cos x\cos y$ is self-adjoint, $\lVert u(t)\rVert_{L^2}$ must be constant in time. This is a much stronger check than eyeballing the plots: it validates the whole chain *exponential symbol $\to$ asymptotic composition $\to$ Peetre decomposition $\to$ FFT application*.


In [ ]:
dxg, dyg = x_grid[1] - x_grid[0], y_grid[1] - y_grid[0]
mass = np.array([np.sum(np.abs(U_arr[i])**2) * dxg * dyg
                 for i in range(U_arr.shape[0])])
rel = mass / mass[0]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(t_arr, rel, lw=2, color='tab:blue')
ax[0].axhline(1.0, color='k', ls='--', lw=1)
ax[0].set_xlabel('t'); ax[0].set_ylabel(r'$\|u(t)\|_2^2 / \|u(0)\|_2^2$')
ax[0].set_title(r'$L^2$ mass conservation'); ax[0].grid(True, alpha=0.3)

ax[1].semilogy(t_arr, np.abs(rel - 1.0) + 1e-16, lw=2, color='tab:red')
ax[1].set_xlabel('t'); ax[1].set_ylabel('relative mass drift')
ax[1].set_title('drift (log scale)'); ax[1].grid(True, alpha=0.3)

fig.tight_layout(); plt.show()

print(f'max relative mass drift = {np.max(np.abs(rel - 1.0)):.3e}')


In [ ]:
# psiop ships animate_scalar_1d but no 2D equivalent, so build one inline.
from matplotlib.animation import FuncAnimation

field = np.real(U_arr)                      # switch to np.abs(U_arr) for amplitude
vmax  = float(np.max(np.abs(field)))

fig, ax = plt.subplots(figsize=(6.5, 5.6))
pcm = ax.pcolormesh(x_grid, y_grid, field[0].T, shading='auto',
                    cmap='RdBu_r', vmin=-vmax, vmax=vmax)
fig.colorbar(pcm, ax=ax, label=r'$\mathrm{Re}\,u$')
ax.set_aspect('equal')
ax.set_xlabel('x'); ax.set_ylabel('y')
title = ax.set_title(f't = {t_arr[0]:.3f}')

def update(i):
    pcm.set_array(field[i].T.ravel())
    title.set_text(f't = {t_arr[i]:.3f}')
    return pcm, title

anim = FuncAnimation(fig, update, frames=len(t_arr), interval=60, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())


---

### Optional further experiments in 2D

* **Joint residual / fast backends.** Swap in `symbol_expr = sp.exp(sp.I*(x*xi + y*eta))*(xi**2 + eta**2)` (NUFFT-representable) or `1/(1 + (x-xi)**2 + (y-eta)**2)` (AAA-representable) at the top of the notebook and rerun `op.print_peetre_decomposition(joint_backend='auto')`. These symbols are *not* separable, so they land in the `joint_residual` bucket and exercise `_auto_select_joint_backend` -> `nufft` / `aaa` / `lowrank`. Compare `op.apply(...)` with `op.apply_hybrid(...)`.
* **Wave equation.** `solve_second_order(-(xi**2 + eta**2), [x, y], f, g, dt, n_steps)` block-diagonalises $u_{tt}=\Delta u$ into a first-order system and returns `(t, U, V, grids)`; visualise with `plot_wave_solution_1d`-style panels.
* **Weyl quantisation.** Recreate `op` with `quantization='weyl'`; `apply_peetre` will first convert via `weyl_to_kn_symbol(order=weyl_order)` and then decompose.
